# Phase 4 — Model Training and Comparison

Trains and compares 6 classifiers on `train_pool`, using **5-fold cross-validation grouped by `Patient File No.`** (not plain K-fold) — a plain split would let augmented copies of the same patient land on both sides of a fold, inflating every score (the Phase 1 finding). Reports both the training-fold and validation-fold score for every model, so overfitting is visible directly rather than assumed.

`holdout_validation` is **not touched in this notebook**. It stays frozen until a final model is selected and tuned (Phase 6/7).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.config import INTERIM_DIR, TABLES_DIR, PATIENT_ID_COL
from src.preprocessing import split_features_target
from src.train import run_all_models_cv, summarize_cv_results, MODEL_SPECS

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

train_pool = pd.read_csv(INTERIM_DIR / "train_pool.csv")
X, y = split_features_target(train_pool)
groups = train_pool[PATIENT_ID_COL]
print("X:", X.shape, "| patients (groups):", groups.nunique())

## Models and why each is included

| Model | Why included |
|---|---|
| **Dummy (most-frequent)** | The floor. Always predicts "No PCOS" — establishes what accuracy looks like with zero information, so later accuracy numbers can be judged against it rather than in isolation |
| **Logistic Regression** | The interpretable baseline. Coefficients are directly readable; if it performs close to the ensembles, simplicity wins |
| **Decision Tree** | A single tree, unrestricted depth. Expected to overfit badly — included specifically to *show* what overfitting looks like in the train-vs-val gap, motivating why Phase 6 will tune its depth |
| **Random Forest** | Bagged trees — usually more robust than a single tree, handles the mixed numeric/binary/categorical feature set without needing feature scaling to matter |
| **Gradient Boosting** (scikit-learn) | Sequential boosting, often the strongest tabular-data performer; no `class_weight` param, so balanced sample weights are computed per training fold and passed in manually |
| **XGBoost** | A faster, regularized boosting implementation; imbalance handled via `scale_pos_weight`, computed per fold from that fold's own class ratio (not a project-wide constant) |

**SVM was not included.** For a dataset this size with a mixed numeric/binary/categorical feature set and where tree ensembles already handle non-linearity well, SVM would mainly add probability-calibration complexity without an expected performance edge over Random Forest/Gradient Boosting — can be added later if there's a specific reason to want it.

All models use `class_weight="balanced"` where the estimator supports it, or an equivalent per-fold sample weighting/`scale_pos_weight` where it doesn't (Phase 3 decision — see `notebooks/03_preprocessing.ipynb` for why SMOTE was not used instead).

In [ ]:
long_results = run_all_models_cv(X, y, groups)
long_results.to_csv(TABLES_DIR / "phase4_cv_long_results.csv", index=False)

summary = summarize_cv_results(long_results)
summary.to_csv(TABLES_DIR / "phase4_cv_model_comparison.csv")
summary.round(3)

## Results

| Model | Val accuracy | Val F1 | Val ROC-AUC | Train ROC-AUC | Train-val ROC-AUC gap |
|---|---:|---:|---:|---:|---:|
| Dummy baseline | 0.694 | 0.000 | 0.500 | 0.500 | 0.000 |
| Decision Tree | 0.761 | 0.612 | 0.725 | 1.000 | **0.275 (severe)** |
| Logistic Regression | 0.851 | 0.767 | 0.918 | 0.979 | 0.061 (small) |
| XGBoost | 0.863 | 0.775 | 0.928 | 1.000 | 0.072 |
| Gradient Boosting | 0.880 | 0.804 | 0.943 | 1.000 | 0.057 |
| **Random Forest** | **0.898** | **0.818** | **0.950** | 1.000 | 0.050 |

(exact numbers reproduced from the cell above; this table is for readability)

**Signs of overfitting:**
- **Decision Tree** is the clear overfitting case: perfect training score (1.000 across every metric) collapsing to 0.725 validation ROC-AUC — a 0.275 gap. An unrestricted tree memorizes the training patients. This motivates depth/leaf-size tuning in Phase 6, or dropping it in favor of the ensembles.
- **Random Forest, Gradient Boosting, XGBoost** all show perfect training scores too (expected — unpruned trees inside an ensemble still memorize), but their validation scores stay much higher (0.928-0.950 ROC-AUC) because averaging/boosting across many trees generalizes better than any single one. Gap size: Random Forest (0.050) < Gradient Boosting (0.057) < XGBoost (0.072).
- **Logistic Regression** has the smallest gap of any non-dummy model (0.061) and a respectable 0.918 validation ROC-AUC — the most "honest" model in the sense that its training score is a good predictor of its validation score. That said, its validation ROC-AUC still trails the ensembles.
- **Dummy baseline** confirms why accuracy is a misleading headline metric here: 69.4% accuracy while catching **zero** PCOS cases (precision/recall/F1 all 0.000). Any model needs to clear this floor on recall/F1/ROC-AUC, not just accuracy.

**Preliminary read (subject to Phase 5's fuller metric suite and Phase 6 tuning):** Random Forest currently leads on validation F1 and ROC-AUC with the smallest ensemble overfitting gap; Gradient Boosting is close behind; Logistic Regression is the strongest *interpretable* option if simplicity is prioritized over the last few points of ROC-AUC. Nothing here is final — Phase 5 adds confusion matrices, PR-AUC, calibration, and specificity, and Phase 6 tunes the promising candidates before any final selection.